In [1]:
!pip install faiss-cpu transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.4 MB/s eta 0:00:00


In [2]:
import os, json, zipfile
import numpy as np
import pandas as pd
import faiss
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Extract Phase 1
p1_zip = "/kaggle/input/notebooks/rajansamar/yolo-clip/_output_.zip"
p1_dir = Path("/kaggle/working/phase1_outputs")
if not p1_dir.exists():
    with zipfile.ZipFile(p1_zip) as z:
        z.extractall(p1_dir)
    print("Phase 1 extracted")

# Extract Phase 2
p2_zip = "/kaggle/input/notebooks/rajansamar/offline/_output_.zip"
p2_dir = Path("/kaggle/working/phase2_outputs")
if not p2_dir.exists():
    with zipfile.ZipFile(p2_zip) as z:
        z.extractall(p2_dir)
    print("Phase 2 extracted")

CROPS_DIR = p1_dir / "crops"
MANIFEST  = p1_dir / "crops_manifest.csv"
DESC_FILE = Path("/kaggle/input/datasets/rajansamar/final-project/VR/Anno/list_description_inshop.json")
WORK_DIR  = Path("/kaggle/working/phase3_finetune")
WORK_DIR.mkdir(exist_ok=True)

print(f"Crops dir exists: {CROPS_DIR.exists()}")
print(f"Manifest exists:  {MANIFEST.exists()}")
print("All paths ready")

Device: cuda
Phase 1 extracted
Phase 2 extracted
Crops dir exists: True
Manifest exists:  True
All paths ready


In [3]:
df = pd.read_csv(MANIFEST)
df["crop_path"] = df["crop_path"].str.replace(
    "/kaggle/working/crops/",
    str(CROPS_DIR) + "/",
    regex=False
)

with open(DESC_FILE) as f:
    desc_list = json.load(f)

caption_map = {}
for entry in desc_list:
    item_id = entry["item"]
    color   = entry.get("color", "")
    desc    = entry.get("description", [])
    text    = f"{color}. " + " ".join(desc[:2])
    caption_map[item_id] = text[:200]

train_df = df[df["split"] == "train"].reset_index(drop=True)
train_df["caption"] = train_df["item_id"].apply(
    lambda x: caption_map.get(x, "a clothing item")
)

# Keep only items with 2+ images (need positives for contrastive loss)
counts    = train_df["item_id"].value_counts()
valid_ids = counts[counts >= 2].index
train_df  = train_df[train_df["item_id"].isin(valid_ids)].reset_index(drop=True)

print(f"Train images: {len(train_df)}")
print(f"Unique items: {train_df['item_id'].nunique()}")

# Verify a path exists
print(f"Sample path exists: {os.path.exists(train_df['crop_path'].iloc[0])}")

Train images: 25870
Unique items: 3985
Sample path exists: True


In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(MODEL_NAME)

# Freeze everything
for param in clip_model.parameters():
    param.requires_grad = False

# Unfreeze last 4 vision encoder blocks
for layer in clip_model.vision_model.encoder.layers[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze visual projection
for param in clip_model.visual_projection.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in clip_model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Trainable: 28,744,704 / 151,277,313 params


In [5]:
class FashionDataset(Dataset):
    def __init__(self, df):
        self.df       = df.reset_index(drop=True)
        unique_ids    = sorted(self.df["item_id"].unique())
        self.id2label = {iid: i for i, iid in enumerate(unique_ids)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row.crop_path).convert("RGB")
        label = self.id2label[row.item_id]
        return image, label

def collate_fn(batch):
    images, labels = zip(*batch)
    inputs = processor(images=list(images), return_tensors="pt")
    return inputs["pixel_values"], torch.tensor(labels, dtype=torch.long)

dataset    = FashionDataset(train_df)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True,
                        num_workers=2, collate_fn=collate_fn, pin_memory=True)

print(f"Batches per epoch: {len(dataloader)}")

Batches per epoch: 405


In [6]:
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temp = temperature

    def forward(self, embeddings, labels):
        N      = embeddings.shape[0]
        sim    = torch.matmul(embeddings, embeddings.T) / self.temp

        mask_self = torch.eye(N, dtype=torch.bool, device=embeddings.device)
        mask_pos  = (labels.unsqueeze(1) == labels.unsqueeze(0)) & ~mask_self

        has_pos = mask_pos.any(dim=1)
        if not has_pos.any():
            return torch.tensor(0.0, requires_grad=True, device=embeddings.device)

        sim      = sim[has_pos]
        mask_pos = mask_pos[has_pos]
        mask_self_f = mask_self[has_pos]

        sim_no_self = sim.masked_fill(mask_self_f, float("-inf"))
        log_denom   = torch.logsumexp(sim_no_self, dim=1, keepdim=True)
        log_prob    = sim - log_denom
        loss        = -(log_prob * mask_pos.float()).sum(dim=1) / \
                       mask_pos.float().sum(dim=1).clamp(min=1)
        return loss.mean()

criterion = SupervisedContrastiveLoss(temperature=0.07)
print("Loss function ready")

Loss function ready


In [7]:
EPOCHS = 10
LR     = 1e-5

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clip_model.parameters()),
    lr=LR, weight_decay=1e-4
)
clip_model.train()
best_loss = float("inf")

for epoch in range(EPOCHS):
    total_loss, steps = 0, 0

    for pixel_values, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        pixel_values = pixel_values.to(DEVICE)
        labels       = labels.to(DEVICE)

        optimizer.zero_grad()
        out  = clip_model.vision_model(pixel_values=pixel_values)
        embs = clip_model.visual_projection(out.pooler_output)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        loss = criterion(embs, labels)

        if not loss.requires_grad:
            continue

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps      += 1

    if steps == 0:
        print(f"Epoch {epoch+1} | No valid batches")
        continue

    avg_loss = total_loss / steps
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        clip_model.save_pretrained(str(WORK_DIR / "clip_finetuned"))
        processor.save_pretrained(str(WORK_DIR  / "clip_finetuned"))
        print(f"  → Best model saved (loss={best_loss:.4f})")

Epoch 1/10: 100%|██████████| 405/405 [01:41<00:00,  3.98it/s]

Epoch 1 | Loss: 1.1254


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best model saved (loss=1.1254)


Epoch 2/10: 100%|██████████| 405/405 [01:56<00:00,  3.47it/s]

...

Epoch 9 | Loss: 0.4186


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best model saved (loss=0.4186)


Epoch 10/10: 100%|██████████| 405/405 [01:57<00:00,  3.46it/s]

Epoch 10 | Loss: 0.4672


In [8]:
ft_model     = CLIPModel.from_pretrained(str(WORK_DIR / "clip_finetuned")).to(DEVICE)
ft_processor = CLIPProcessor.from_pretrained(str(WORK_DIR / "clip_finetuned"))
ft_model.eval()

# Load gallery/query metadata from Phase 2
gallery_df = pd.read_csv(p2_dir / "phase2" / "gallery_meta.csv")
query_df   = pd.read_csv(p2_dir / "phase2" / "query_meta.csv")

gallery_df["crop_path"] = gallery_df["crop_path"].str.replace(
    "/kaggle/working/crops/", str(CROPS_DIR) + "/", regex=False)
query_df["crop_path"]   = query_df["crop_path"].str.replace(
    "/kaggle/working/crops/", str(CROPS_DIR) + "/", regex=False)

gallery_df["caption"] = gallery_df["item_id"].apply(
    lambda x: caption_map.get(x, "a clothing item"))
query_df["caption"]   = query_df["item_id"].apply(
    lambda x: caption_map.get(x, "a clothing item"))

BATCH_SIZE = 128

def embed_images_ft(paths):
    all_embs = []
    for i in tqdm(range(0, len(paths), BATCH_SIZE), desc="Image embeds"):
        images = [Image.open(p).convert("RGB") for p in paths[i:i+BATCH_SIZE]]
        inputs = ft_processor(images=images, return_tensors="pt")
        pv     = inputs["pixel_values"].to(DEVICE)
        with torch.no_grad():
            out  = ft_model.vision_model(pixel_values=pv)
            embs = ft_model.visual_projection(out.pooler_output)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs).astype("float32")

def embed_texts_ft(texts):
    all_embs = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Text embeds"):
        batch  = texts[i:i+BATCH_SIZE]
        inputs = ft_processor(text=batch, return_tensors="pt",
                              padding=True, truncation=True, max_length=77)
        iids   = inputs["input_ids"].to(DEVICE)
        amask  = inputs["attention_mask"].to(DEVICE)
        with torch.no_grad():
            out  = ft_model.text_model(input_ids=iids, attention_mask=amask)
            embs = ft_model.text_projection(out.pooler_output)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs).astype("float32")

print("Re-embedding gallery...")
gallery_img_ft = embed_images_ft(gallery_df["crop_path"].tolist())
gallery_txt_ft = embed_texts_ft(gallery_df["caption"].tolist())

print("Re-embedding query...")
query_img_ft   = embed_images_ft(query_df["crop_path"].tolist())
query_txt_ft   = embed_texts_ft(query_df["caption"].tolist())

np.save(WORK_DIR / "gallery_img_ft.npy", gallery_img_ft)
np.save(WORK_DIR / "gallery_txt_ft.npy", gallery_txt_ft)
np.save(WORK_DIR / "query_img_ft.npy",   query_img_ft)
np.save(WORK_DIR / "query_txt_ft.npy",   query_txt_ft)
print("Embeddings saved")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Re-embedding gallery...


Text embeds: 100%|██████████| 99/99 [00:21<00:00,  4.60it/s]


Re-embedding query...


Text embeds: 100%|██████████| 112/112 [00:24<00:00,  4.63it/s]

Embeddings saved


In [9]:
def fuse(img_embs, txt_embs, alpha):
    fused = alpha * img_embs + (1 - alpha) * txt_embs
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    return (fused / norms).astype("float32")

def build_hnsw_index(embeddings, M=32):
    dim   = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 128
    index.add(embeddings)
    return index

def recall_at_k(query_ids, gallery_ids, retrieved_indices, k):
    hits = 0
    for i, qid in enumerate(query_ids):
        if qid in [gallery_ids[j] for j in retrieved_indices[i][:k]]:
            hits += 1
    return hits / len(query_ids)

def ndcg_at_k(query_ids, gallery_ids, retrieved_indices, k):
    g_arr = np.array(gallery_ids)
    scores = []
    for i, qid in enumerate(query_ids):
        top_k = retrieved_indices[i][:k]
        gains = [1 if gallery_ids[j] == qid else 0 for j in top_k]
        dcg   = sum(g / np.log2(r+2) for r, g in enumerate(gains))
        n_rel = int(np.sum(g_arr == qid))
        idcg  = sum(1 / np.log2(r+2) for r in range(min(n_rel, k)))
        scores.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(scores)

def map_at_k(query_ids, gallery_ids, retrieved_indices, k):
    aps = []
    for i, qid in enumerate(query_ids):
        top_k = retrieved_indices[i][:k]
        hits, prec_sum = 0, 0
        for r, j in enumerate(top_k):
            if gallery_ids[j] == qid:
                hits += 1
                prec_sum += hits / (r+1)
        aps.append(prec_sum / max(hits, 1) if hits > 0 else 0)
    return np.mean(aps)

def evaluate(query_ids, gallery_ids, retrieved_indices, label=""):
    print(f"\n{'='*45}")
    print(f"  {label}")
    print(f"{'='*45}")
    results = []
    for k in [5, 10, 15]:
        r = recall_at_k(query_ids, gallery_ids, retrieved_indices, k)
        n = ndcg_at_k(query_ids, gallery_ids, retrieved_indices, k)
        m = map_at_k(query_ids, gallery_ids, retrieved_indices, k)
        print(f"  K={k:2d} | Recall={r:.4f} | NDCG={n:.4f} | mAP={m:.4f}")
        results.append({"config": label, "K": k, "Recall": r, "NDCG": n, "mAP": m})
    return results

query_ids   = query_df["item_id"].tolist()
gallery_ids = gallery_df["item_id"].tolist()
all_results = []

for alpha in [0.7, 0.5]:
    gallery_fused  = fuse(gallery_img_ft, gallery_txt_ft, alpha)
    query_fused    = fuse(query_img_ft,   query_txt_ft,   alpha)
    index_C        = build_hnsw_index(gallery_fused)
    _, retrieved_C = index_C.search(query_fused, 15)
    res = evaluate(query_ids, gallery_ids, retrieved_C,
                   label=f"C: Fine-tuned CLIP (α={alpha})")
    all_results.extend(res)
    faiss.write_index(index_C, str(WORK_DIR / f"index_C_alpha{int(alpha*10)}.faiss"))

results_df = pd.DataFrame(all_results)
results_df.to_csv(WORK_DIR / "ablation_C_results.csv", index=False)
print("\n=== ABLATION C RESULTS ===")
print(results_df.to_string(index=False))


  C: Fine-tuned CLIP (α=0.7)
  K= 5 | Recall=0.8742 | NDCG=0.6439 | mAP=0.7732
  K=10 | Recall=0.9071 | NDCG=0.6461 | mAP=0.7415
  K=15 | Recall=0.9232 | NDCG=0.6555 | mAP=0.7192

  C: Fine-tuned CLIP (α=0.5)
  K= 5 | Recall=0.9793 | NDCG=0.8957 | mAP=0.9429
  K=10 | Recall=0.9859 | NDCG=0.8949 | mAP=0.9262
  K=15 | Recall=0.9890 | NDCG=0.8988 | mAP=0.9149

=== ABLATION C RESULTS ===
                    config  K   Recall     NDCG      mAP
C: Fine-tuned CLIP (α=0.7)  5 0.874244 0.643863 0.773196
C: Fine-tuned CLIP (α=0.7) 10 0.907090 0.646071 0.741531
C: Fine-tuned CLIP (α=0.7) 15 0.923196 0.655482 0.719172
C: Fine-tuned CLIP (α=0.5)  5 0.979252 0.895720 0.942902
C: Fine-tuned CLIP (α=0.5) 10 0.985933 0.894910 0.926247
C: Fine-tuned CLIP (α=0.5) 15 0.989028 0.898838 0.914950


In [10]:
print("Phase 4 complete.")
print(f"Outputs saved to: {WORK_DIR}")
print(os.listdir(WORK_DIR))

Phase 4 complete.
Outputs saved to: /kaggle/working/phase3_finetune
['query_img_ft.npy', 'clip_finetuned', 'gallery_img_ft.npy', 'gallery_txt_ft.npy', 'index_C_alpha5.faiss', 'index_C_alpha7.faiss', 'query_txt_ft.npy', 'ablation_C_results.csv']


In [11]:
import numpy as np

# Load Phase 2 embeddings (for Ablation B)
gallery_img_embs = np.load(str(p2_dir / "phase2" / "gallery_img_embs.npy"))
gallery_txt_embs = np.load(str(p2_dir / "phase2" / "gallery_txt_embs.npy"))
query_img_embs   = np.load(str(p2_dir / "phase2" / "query_img_embs.npy"))
query_txt_embs   = np.load(str(p2_dir / "phase2" / "query_txt_embs.npy"))

print("Phase 2 embeddings loaded")

SEEDS = [616, 624, 572]

def evaluate_with_seed(seed, g_img, g_txt, q_img, q_txt,
                       query_ids, gallery_ids, alpha):
    np.random.seed(seed)
    gallery_fused = fuse(g_img, g_txt, alpha)
    query_fused   = fuse(q_img, q_txt, alpha)
    index         = build_hnsw_index(gallery_fused)
    _, retrieved  = index.search(query_fused, 15)
    res = {}
    for k in [5, 10, 15]:
        res[f"Recall@{k}"] = recall_at_k(query_ids, gallery_ids, retrieved, k)
        res[f"NDCG@{k}"]   = ndcg_at_k(query_ids, gallery_ids, retrieved, k)
        res[f"mAP@{k}"]    = map_at_k(query_ids, gallery_ids, retrieved, k)
    return res

configs = [
    ("B", 0.7, gallery_img_embs, gallery_txt_embs, query_img_embs, query_txt_embs),
    ("B", 0.5, gallery_img_embs, gallery_txt_embs, query_img_embs, query_txt_embs),
    ("C", 0.7, gallery_img_ft,   gallery_txt_ft,   query_img_ft,   query_txt_ft),
    ("C", 0.5, gallery_img_ft,   gallery_txt_ft,   query_img_ft,   query_txt_ft),
]

final_rows = []

for ablation, alpha, g_img, g_txt, q_img, q_txt in configs:
    label        = f"{ablation} (α={alpha})"
    seed_records = []

    for seed in SEEDS:
        res = evaluate_with_seed(seed, g_img, g_txt, q_img, q_txt,
                                 query_ids, gallery_ids, alpha)
        seed_records.append(res)

    print(f"\n{'='*50}")
    print(f"  Config {label} — mean ± std over seeds {SEEDS}")
    print(f"{'='*50}")
    for k in [5, 10, 15]:
        for metric in ["Recall", "NDCG", "mAP"]:
            vals = [r[f"{metric}@{k}"] for r in seed_records]
            mean = np.mean(vals)
            std  = np.std(vals)
            print(f"  K={k:2d} | {metric}={mean:.4f} ± {std:.4f}")
            final_rows.append({
                "config": label, "K": k,
                "metric": metric, "mean": mean, "std": std
            })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(WORK_DIR / "ablation_mean_std.csv", index=False)
print("\nSaved to ablation_mean_std.csv")

Phase 2 embeddings loaded

  Config B (α=0.7) — mean ± std over seeds [616, 624, 572]
  K= 5 | Recall=0.8516 ± 0.0000
  K= 5 | NDCG=0.6134 ± 0.0000
  K= 5 | mAP=0.7582 ± 0.0000
  K=10 | Recall=0.8784 ± 0.0000
  K=10 | NDCG=0.6086 ± 0.0000
  K=10 | mAP=0.7286 ± 0.0000
  K=15 | Recall=0.8919 ± 0.0000
  K=15 | NDCG=0.6142 ± 0.0000
  K=15 | mAP=0.7089 ± 0.0000

  Config B (α=0.5) — mean ± std over seeds [616, 624, 572]
  K= 5 | Recall=0.9896 ± 0.0000
  K= 5 | NDCG=0.9398 ± 0.0000
  K= 5 | mAP=0.9677 ± 0.0000
  K=10 | Recall=0.9943 ± 0.0000
  K=10 | NDCG=0.9428 ± 0.0000
  K=10 | mAP=0.9548 ± 0.0000
  K=15 | Recall=0.9959 ± 0.0000
  K=15 | NDCG=0.9462 ± 0.0000
  K=15 | mAP=0.9469 ± 0.0000

  Config C (α=0.7) — mean ± std over seeds [616, 624, 572]
  K= 5 | Recall=0.8742 ± 0.0000
  K= 5 | NDCG=0.6439 ± 0.0000
  K= 5 | mAP=0.7732 ± 0.0000
  K=10 | Recall=0.9071 ± 0.0000
  K=10 | NDCG=0.6461 ± 0.0000
  K=10 | mAP=0.7415 ± 0.0000
  K=15 | Recall=0.9232 ± 0.0000
  K=15 | NDCG=0.6555 ± 0.0000
  K=